In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data"

CDR_PATH = DATA_PATH / "raw_cdr" / "clean_cdr.parquet"

In [3]:
all_cdr = pd.read_parquet(CDR_PATH)
all_cdr["profile_date"] = pd.to_datetime(
    all_cdr["answer_time"]
).dt.date

print(f"Shape: {all_cdr.shape}")
print(all_cdr.columns.tolist())

Shape: (7213036, 10)
['msisdn', 'answer_time', 'duration_in_seconds', 'lac', 'cell', 'calling_number', 'called_number', 'record_type', 'label', 'profile_date']


In [4]:
diagnostics = []

for day, day_df in all_cdr.groupby("profile_date"):
    msisdns = day_df["msisdn"].nunique()

    outgoing_msisdns = day_df.loc[
        day_df["record_type"] == 0,
        "msisdn"
    ].nunique()

    calling_numbers = day_df["calling_number"].nunique()

    diagnostics.append({
        "day": day,
        "all_msisdns": msisdns,
        "outgoing_msisdns": outgoing_msisdns,
        "calling_numbers": calling_numbers
    })

diagnostics_df = pd.DataFrame(diagnostics)

display(diagnostics_df)

,day,all_msisdns,outgoing_msisdns,calling_numbers
0,2022-11-01,28004,23840,133904
1,2022-11-02,28401,24631,141241
2,2022-11-03,28501,24322,140289
3,2022-11-04,28719,24434,144210
4,2022-11-05,28648,24454,142929
5,2022-11-06,28468,24125,128233
6,2022-11-07,28916,24643,145675
7,2022-11-08,28771,24470,144216
8,2022-11-09,28761,24099,142345
9,2022-11-10,28535,23778,136604


In [10]:
# Normalize phone-number columns to the same representation
normalized = {}

for col in ["msisdn", "calling_number", "called_number"]:
    normalized[col] = (
        pd.to_numeric(all_cdr[col], errors="coerce")
        .dropna()
        .astype("int64")
        .astype(str)
    )

    print(
        f"{col:20s} "
        f"unique = {normalized[col].nunique():,}"
    )

print(
    "MSISDN ∩ calling_number:",
    len(
        set(normalized["msisdn"])
        & set(normalized["calling_number"])
    )
)

print(
    "MSISDN ∩ called_number:",
    len(
        set(normalized["msisdn"])
        & set(normalized["called_number"])
    )
)

msisdn               unique = 31,742
calling_number       unique = 849,347
called_number        unique = 1,143,615
MSISDN ∩ calling_number: 31086
MSISDN ∩ called_number: 31510


In [9]:
all_msisdns = set(
    pd.to_numeric(all_cdr["msisdn"], errors="coerce")
    .dropna()
    .astype("int64")
    .astype(str)
)

diagnostics = []

for day, day_df in all_cdr.groupby("profile_date"):

    calling_numbers = set(
        pd.to_numeric(
            day_df["calling_number"],
            errors="coerce"
        )
        .dropna()
        .astype("int64")
        .astype(str)
    )

    matching = calling_numbers & all_msisdns

    diagnostics.append({
        "day": day,
        "unique_calling_numbers": len(calling_numbers),
        "calling_numbers_in_global_msisdns": len(matching),
        "percentage_matching": (
            len(matching) / len(calling_numbers) * 100
        )
    })

calling_msisdn_diagnostic = (
    pd.DataFrame(diagnostics)
    .sort_values("day")
    .reset_index(drop=True)
)

display(calling_msisdn_diagnostic)

,day,unique_calling_numbers,calling_numbers_in_global_msisdns,percentage_matching
0,2022-11-01,133904,23840,17.803800
1,2022-11-02,141241,24631,17.438987
2,2022-11-03,140289,24322,17.337068
3,2022-11-04,144210,24434,16.943347
4,2022-11-05,142929,24454,17.109194
5,2022-11-06,128233,24125,18.813410
6,2022-11-07,145675,24643,16.916424
7,2022-11-08,144216,24470,16.967604
8,2022-11-09,142345,24099,16.929994
9,2022-11-10,136604,23778,17.406518
